# Backfill Gold — Orquestador por Corte

## Propósito

Reprocesar la capa Gold corte por corte, en orden cronológico,
pasando `fecha_fin_corte_filtro` a cada notebook para evitar
window overlap.

## Cuándo usarlo

- Primera carga después de implementar Etapa C (reprocesabilidad)
- Corrección de un corte específico (`fecha_inicio` = `fecha_fin`)
- Reconstrucción completa de Gold desde Silver

## Orden de ejecución por corte

1. Dimensiones (secuencial): locations → fecha → shipping_mode → customers → products
2. Hechos: fact_orders → fact_order_details → order_summary
3. KPIs (secuencial): order_detail → executive_summary → region_detail → alerts
4. Reporte Excel / correo

## Prerequisito

Silver debe estar correcto y completo antes de correr este notebook.

In [0]:
# ============================================================
# BACKFILL_GOLD — Celda 1: Parámetros
# ============================================================

dbutils.widgets.text("silver_catalog",   "slv_dataco",    "Catálogo Silver")
dbutils.widgets.text("silver_schema",    "supply_chain",  "Esquema Silver")
dbutils.widgets.text("gold_catalog",     "gld_dataco",    "Catálogo Gold")
dbutils.widgets.text("gold_schema",      "supply_chain",  "Esquema Gold")
dbutils.widgets.text("metadata_catalog", "metadata_dataco", "Catálogo Metadata")
dbutils.widgets.text("metadata_schema",  "supply_chain",  "Esquema Metadata")

# Rango de cortes a reprocesar.
# Dejar vacíos = reprocesa TODOS los cortes de Silver.
# Pasar el mismo valor en ambos = reprocesa un único corte.
dbutils.widgets.text("fecha_inicio_backfill", "", "Primer corte a reprocesar (YYYY-MM-DD, vacío = desde el primero)")
dbutils.widgets.text("fecha_fin_backfill",    "", "Último corte a reprocesar  (YYYY-MM-DD, vacío = hasta el último)")

silver_catalog   = dbutils.widgets.get("silver_catalog")
silver_schema    = dbutils.widgets.get("silver_schema")
gold_catalog     = dbutils.widgets.get("gold_catalog")
gold_schema      = dbutils.widgets.get("gold_schema")
metadata_catalog = dbutils.widgets.get("metadata_catalog")
metadata_schema  = dbutils.widgets.get("metadata_schema")

fecha_inicio_raw = dbutils.widgets.get("fecha_inicio_backfill").strip() or None
fecha_fin_raw    = dbutils.widgets.get("fecha_fin_backfill").strip()    or None

print(f"Silver : {silver_catalog}.{silver_schema}")
print(f"Gold   : {gold_catalog}.{gold_schema}")
print(f"Rango  : {fecha_inicio_raw or 'primer corte'} → {fecha_fin_raw or 'ultimo corte'}")

In [0]:
# ============================================================
# BACKFILL_GOLD — Celda 2: Obtener cortes a reprocesar
# ============================================================

from pyspark.sql import functions as F

SILVER_PEDIDOS_FQN = f"{silver_catalog}.{silver_schema}.hd_pedidos"

df_cortes = (
    spark.table(SILVER_PEDIDOS_FQN)
    .filter("estado_calidad = 'VALIDO'")
    .select("fecha_fin_corte")
    .distinct()
    .filter(F.col("fecha_fin_corte").isNotNull())
)

if fecha_inicio_raw:
    df_cortes = df_cortes.filter(
        F.col("fecha_fin_corte") >= F.lit(fecha_inicio_raw).cast("date")
    )
if fecha_fin_raw:
    df_cortes = df_cortes.filter(
        F.col("fecha_fin_corte") <= F.lit(fecha_fin_raw).cast("date")
    )

cortes = sorted([r["fecha_fin_corte"].strftime("%Y-%m-%d") for r in df_cortes.collect()])

print(f"Cortes a reprocesar: {len(cortes)}")
for c in cortes:
    print(f"  {c}")

In [0]:
# ============================================================
# BACKFILL_GOLD — Celda 3: Parámetros comunes para dbutils.notebook.run
# ============================================================

PARAMS_BASE = {
    "silver_catalog":      silver_catalog,
    "silver_schema":       silver_schema,
    "gold_catalog":        gold_catalog,
    "gold_schema":         gold_schema,
    "metadata_catalog":    metadata_catalog,
    "metadata_schema":     metadata_schema,
    "id_ejecucion_manual": "backfill",
}

# Ruta base de los notebooks Gold (relativa a este notebook)
BASE_DIMS  = "../03_gold_dimensiones/01_negocio"
BASE_HEC   = "../03_gold_hechos/01_negocio"
BASE_KPI   = "../03_gold_kpis/01_negocio"
BASE_REP   = "../04_reporte_excel_correo/01_negocio"

TIMEOUT_SEG = 1800  # 30 minutos por notebook

print("Configuracion lista")

In [0]:
# ============================================================
# BACKFILL_GOLD — Celda 4: Loop por corte
# ============================================================

errores = []

for corte in cortes:
    print(f"\n{'='*60}")
    print(f"CORTE: {corte}")
    print(f"{'='*60}")

    params = {**PARAMS_BASE, "fecha_fin_corte_filtro": corte}

    # ----------------------------------------------------------
    # 1. DIMENSIONES (secuencial)
    # Orden: las que no dependen de nada van primero.
    # En un Job podrían correr en paralelo como tasks.
    # ----------------------------------------------------------
    dims = [
        ("locations",     f"{BASE_DIMS}/gld_dim_locations"),
        ("fecha",         f"{BASE_DIMS}/gld_dim_fecha"),
        ("shipping_mode", f"{BASE_DIMS}/gld_dim_shipping_mode"),
        ("customers",     f"{BASE_DIMS}/gld_dim_customers"),
        ("products",      f"{BASE_DIMS}/gld_dim_products"),
    ]

    for nombre, path in dims:
        print(f"  dim_{nombre}...", end=" ")
        try:
            dbutils.notebook.run(path, TIMEOUT_SEG, params)
            print("OK")
        except Exception as e:
            msg = f"[{corte}] dim_{nombre}: {e}"
            print(f"ERROR — {msg}")
            errores.append(msg)
            break  # detiene el corte si falla una dimensión
    else:
        # ----------------------------------------------------------
        # 2. HECHOS (secuencial, dependen de dimensiones)
        # ----------------------------------------------------------
        hechos = [
            ("fact_orders",        f"{BASE_HEC}/gld_fact_orders"),
            ("fact_order_details", f"{BASE_HEC}/gld_fact_order_details"),
            ("order_summary",      f"{BASE_HEC}/gld_order_summary"),
        ]

        for nombre, path in hechos:
            print(f"  {nombre}...", end=" ")
            try:
                dbutils.notebook.run(path, TIMEOUT_SEG, params)
                print("OK")
            except Exception as e:
                msg = f"[{corte}] {nombre}: {e}"
                print(f"ERROR — {msg}")
                errores.append(msg)
                break
        else:
            # ----------------------------------------------------------
            # 3. KPIs (secuencial, dependen de hechos)
            # En un Job podrían correr en paralelo como tasks.
            # ----------------------------------------------------------
            kpis = [
                ("kpi_order_detail",       f"{BASE_KPI}/gld_kpi_order_detail"),
                ("kpi_executive_summary",  f"{BASE_KPI}/gld_kpi_executive_summary"),
                ("kpi_region_detail",      f"{BASE_KPI}/gld_kpi_region_detail"),
                ("kpi_alerts",             f"{BASE_KPI}/gld_kpi_alerts"),
            ]

            for nombre, path in kpis:
                print(f"  {nombre}...", end=" ")
                try:
                    dbutils.notebook.run(path, TIMEOUT_SEG, params)
                    print("OK")
                except Exception as e:
                    msg = f"[{corte}] {nombre}: {e}"
                    print(f"ERROR — {msg}")
                    errores.append(msg)
                    break
            else:
                # ----------------------------------------------------------
                # 4. REPORTE EXCEL / CORREO
                # ----------------------------------------------------------
                print(f"  reporte_excel_correo...", end=" ")
                try:
                    dbutils.notebook.run(
                        f"{BASE_REP}/gld_reporte_excel_correo",
                        TIMEOUT_SEG,
                        params
                    )
                    print("OK")
                except Exception as e:
                    msg = f"[{corte}] reporte_excel_correo: {e}"
                    print(f"ERROR — {msg}")
                    errores.append(msg)

print(f"\n{'='*60}")
print(f"BACKFILL TERMINADO")
print(f"Cortes procesados : {len(cortes)}")
print(f"Errores           : {len(errores)}")
for e in errores:
    print(f"  {e}")

In [0]:
# ============================================================
# BACKFILL_GOLD — Celda 5: Resumen de snapshots generados
# ============================================================

from pyspark.sql import functions as F

tablas_kpi = [
    f"{gold_catalog}.{gold_schema}.desc_ft_kpi_resumen_ejecutivo",
    f"{gold_catalog}.{gold_schema}.desc_ft_kpi_detalle_region",
    f"{gold_catalog}.{gold_schema}.desc_ft_kpi_alertas",
]

print("Snapshots en tablas KPI:")
for tabla in tablas_kpi:
    try:
        n = (
            spark.table(tabla)
            .select("fecha_snapshot")
            .distinct()
            .count()
        )
        print(f"  {tabla.split('.')[-1]}: {n} snapshots")
    except Exception as e:
        print(f"  {tabla.split('.')[-1]}: no accesible ({e})")

if errores:
    raise Exception(f"Backfill terminó con {len(errores)} error(es) — ver Celda 4")
else:
    print("\nBackfill completado sin errores")